In [ ]:
# ================================================================
# CGI_MMLP
# DLBCL Pearson Top-K Sensitivity Analysis
#
# K = 50, 100, 200, 500
#
# ================================================================

!pip -q install imbalanced-learn

import time
import itertools
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")


FILE_PATH = "/content/sample_data/DLBCL_Train.csv"
LABEL_COL = "Class"

TOP_K_VALUES = [50, 100, 200, 500]

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS = 5


# ================================================================
# LOAD DATA
# ================================================================

df = pd.read_csv(FILE_PATH)

X = df.drop(columns=[LABEL_COL]).copy()
y_raw = df[LABEL_COL].copy()

X = X.apply(pd.to_numeric, errors="coerce")

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Samples:", X.shape[0])
print("Genes/probes:", X.shape[1])

print("\nClass distribution:")
print(y_raw.value_counts())


# ================================================================
# ENCODE CLASS
# ================================================================

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

print("\nClass mapping:")
for i, cls in enumerate(encoder.classes_):
    print(f"{cls} -> {i}")


# ================================================================
# 80/20 TRAIN-TEST SPLIT
# ================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTraining samples:", len(X_train))
print("Independent test samples:", len(X_test))

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts().sort_index())


# ================================================================
# FIXED 5-FOLD CV
# ================================================================

cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fixed_folds = list(
    cv.split(X_train, y_train)
)

print("\nSame fixed folds used for all K values.")


# ================================================================
# PEARSON FUNCTION
# ================================================================

def absolute_pearson(X_data, y_data):

    X_data = np.asarray(X_data, dtype=np.float64)
    y_data = np.asarray(y_data, dtype=np.float64)

    X_center = X_data - X_data.mean(axis=0)
    y_center = y_data - y_data.mean()

    numerator = X_center.T @ y_center

    denominator = np.sqrt(
        np.sum(X_center ** 2, axis=0)
        *
        np.sum(y_center ** 2)
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        r = numerator / denominator

    r = np.nan_to_num(
        r,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return np.abs(r)


# ================================================================
# JACCARD FUNCTION
# ================================================================

def mean_pairwise_jaccard(gene_sets):

    values = []

    for a, b in itertools.combinations(gene_sets, 2):

        intersection = len(a.intersection(b))
        union = len(a.union(b))

        if union > 0:
            values.append(intersection / union)

    return np.mean(values) if values else np.nan


# ================================================================
# STORAGE
# ================================================================

fold_results = []

selected_gene_sets = {
    K: [] for K in TOP_K_VALUES
}

gene_names = np.array(X_train.columns)


# ================================================================
# RUN ANALYSIS
# ================================================================

for fold_no, (train_idx, val_idx) in enumerate(
    fixed_folds,
    start=1
):

    print("\n" + "=" * 75)
    print(f"FOLD {fold_no}")
    print("=" * 75)

    # ------------------------------------------------------------
    # Fold data
    # ------------------------------------------------------------

    X_fold_train = X_train.iloc[train_idx].copy()
    X_fold_val = X_train.iloc[val_idx].copy()

    y_fold_train = y_train[train_idx]
    y_fold_val = y_train[val_idx]


    # ------------------------------------------------------------
    # Mean imputation
    # ------------------------------------------------------------

    imputer = SimpleImputer(strategy="mean")

    Xtr_imp = imputer.fit_transform(X_fold_train)
    Xval_imp = imputer.transform(X_fold_val)


    # ------------------------------------------------------------
    # Yeo-Johnson
    #
    # standardize=False here because we want transformation first.
    # Scaling for SVM is performed separately AFTER Top-K selection.
    # ------------------------------------------------------------

    yj = PowerTransformer(
        method="yeo-johnson",
        standardize=False
    )

    Xtr_yj = yj.fit_transform(Xtr_imp)
    Xval_yj = yj.transform(Xval_imp)


    # ------------------------------------------------------------
    # Pearson ranking on TRAINING fold only
    # ------------------------------------------------------------

    correlations = absolute_pearson(
        Xtr_yj,
        y_fold_train
    )

    ranked_indices = np.argsort(
        correlations
    )[::-1]

    ranked_correlations = correlations[
        ranked_indices
    ]

    ranked_genes = gene_names[
        ranked_indices
    ]


    # ------------------------------------------------------------
    # Evaluate each K
    # ------------------------------------------------------------

    for K in TOP_K_VALUES:

        top_indices = ranked_indices[:K]

        top_r = ranked_correlations[:K]

        top_genes = ranked_genes[:K]


        # --------------------------------------------------------
        # Pearson statistics
        # --------------------------------------------------------

        mean_r = np.mean(top_r)
        median_r = np.median(top_r)

        # K-th gene's correlation
        cutoff_r = top_r[-1]


        selected_gene_sets[K].append(
            set(top_genes)
        )


        # --------------------------------------------------------
        # Select Top-K features
        # --------------------------------------------------------

        Xtr_K = Xtr_yj[:, top_indices]
        Xval_K = Xval_yj[:, top_indices]


        # --------------------------------------------------------
        # STANDARDIZATION
        #
        # Fit ONLY on training fold.
        #
        # Important:
        # Scaling does NOT change Pearson gene ranking.
        # It is used here to prevent numerical instability
        # in the SVM.
        # --------------------------------------------------------

        scaler = StandardScaler()

        Xtr_K = scaler.fit_transform(
            Xtr_K
        )

        Xval_K = scaler.transform(
            Xval_K
        )


        # --------------------------------------------------------
        # SMOTE - TRAINING FOLD ONLY
        # --------------------------------------------------------

        class_counts = np.bincount(
            y_fold_train
        )

        minority_count = class_counts.min()

        smote_k = min(
            5,
            minority_count - 1
        )

        if smote_k >= 1:

            smote = SMOTE(
                random_state=RANDOM_STATE,
                k_neighbors=smote_k
            )

            Xtr_model, ytr_model = (
                smote.fit_resample(
                    Xtr_K,
                    y_fold_train
                )
            )

        else:

            Xtr_model = Xtr_K
            ytr_model = y_fold_train


        # --------------------------------------------------------
        # START RUNTIME
        #
        # This measures K-specific classifier processing.
        # --------------------------------------------------------

        start_time = time.perf_counter()


        # --------------------------------------------------------
        # LIBLINEAR SVM
        # --------------------------------------------------------

        model = LinearSVC(
            C=1.0,
            tol=1e-3,
            class_weight="balanced",
            max_iter=5000,
            dual="auto",
            random_state=RANDOM_STATE
        )


        model.fit(
            Xtr_model,
            ytr_model
        )


        predictions = model.predict(
            Xval_K
        )


        elapsed_time = (
            time.perf_counter()
            -
            start_time
        )


        # --------------------------------------------------------
        # Accuracy
        # --------------------------------------------------------

        accuracy = accuracy_score(
            y_fold_val,
            predictions
        )


        # --------------------------------------------------------
        # F1
        # --------------------------------------------------------

        f1 = f1_score(
            y_fold_val,
            predictions,
            average="weighted",
            zero_division=0
        )


        # --------------------------------------------------------
        # Save result
        # --------------------------------------------------------

        fold_results.append({

            "Fold":
                fold_no,

            "Top_K":
                K,

            "Mean_Pearson":
                mean_r,

            "Median_Pearson":
                median_r,

            "Correlation_Cutoff":
                cutoff_r,

            "Accuracy":
                accuracy,

            "F1":
                f1,

            "Runtime_sec":
                elapsed_time
        })


        print(
            f"K={K:<3} | "
            f"Mean|r|={mean_r:.4f} | "
            f"Cutoff={cutoff_r:.4f} | "
            f"Acc={accuracy:.4f} | "
            f"F1={f1:.4f} | "
            f"Runtime={elapsed_time:.4f}s"
        )


# ================================================================
# FOLD-LEVEL RESULTS
# ================================================================

fold_df = pd.DataFrame(
    fold_results
)


# ================================================================
# SUMMARY
# ================================================================

summary = []


for K in TOP_K_VALUES:

    temp = fold_df[
        fold_df["Top_K"] == K
    ]

    stability = mean_pairwise_jaccard(
        selected_gene_sets[K]
    )

    summary.append({

        "Top_K":
            K,

        "Mean_Pearson":
            temp["Mean_Pearson"].mean(),

        "Mean_Pearson_SD":
            temp["Mean_Pearson"].std(ddof=1),

        "Median_Pearson":
            temp["Median_Pearson"].mean(),

        "Correlation_Cutoff":
            temp["Correlation_Cutoff"].mean(),

        "Correlation_Cutoff_SD":
            temp["Correlation_Cutoff"].std(ddof=1),

        "Accuracy_Mean":
            temp["Accuracy"].mean(),

        "Accuracy_SD":
            temp["Accuracy"].std(ddof=1),

        "F1_Mean":
            temp["F1"].mean(),

        "F1_SD":
            temp["F1"].std(ddof=1),

        "Jaccard_Stability":
            stability,

        "Runtime_Mean_sec":
            temp["Runtime_sec"].mean(),

        "Runtime_SD_sec":
            temp["Runtime_sec"].std(ddof=1)
    })


summary_df = pd.DataFrame(
    summary
)


# ================================================================
# PUBLICATION TABLE
# ================================================================

publication_table = pd.DataFrame({

    "Top-K Genes":
        summary_df["Top_K"],

    "Mean |r|":
        summary_df[
            "Mean_Pearson"
        ].round(3),

    "Correlation Cutoff":
        summary_df[
            "Correlation_Cutoff"
        ].round(3),

    "Accuracy (mean ± SD)":
        summary_df.apply(
            lambda x:
            f"{x['Accuracy_Mean']:.3f} ± "
            f"{x['Accuracy_SD']:.3f}",
            axis=1
        ),

    "F1-score (mean ± SD)":
        summary_df.apply(
            lambda x:
            f"{x['F1_Mean']:.3f} ± "
            f"{x['F1_SD']:.3f}",
            axis=1
        ),

    "Jaccard Stability":
        summary_df[
            "Jaccard_Stability"
        ].round(3),

    "Runtime (s)":
        summary_df[
            "Runtime_Mean_sec"
        ].round(4)
})


# ================================================================
# DISPLAY
# ================================================================

print("\n")
print("=" * 110)
print("DETAILED RESULTS")
print("=" * 110)

display(
    summary_df.round(4)
)


print("\n")
print("=" * 110)
print("PUBLICATION-READY TABLE")
print("=" * 110)

display(
    publication_table
)


# ================================================================
# SAVE CSV
# ================================================================

fold_df.to_csv(
    "/content/DLBCL_Sensitivity_Fold_Level.csv",
    index=False
)

summary_df.to_csv(
    "/content/DLBCL_Sensitivity_Detailed.csv",
    index=False
)

publication_table.to_csv(
    "/content/DLBCL_Sensitivity_Publication.csv",
    index=False
)


print("\nSaved successfully:")
print("DLBCL_Sensitivity_Fold_Level.csv")
print("DLBCL_Sensitivity_Detailed.csv")
print("DLBCL_Sensitivity_Publication.csv")


# ================================================================
# LATEX TABLE
# ================================================================

print("\nLATEX:")
print(
    publication_table.to_latex(
        index=False,
        escape=False
    )
)

DATASET
Samples: 62
Genes/probes: 2647

Class distribution:
Class
Cancer    46
normal    16
Name: count, dtype: int64

Class mapping:
Cancer -> 0
normal -> 1

Training samples: 49
Independent test samples: 13

Training distribution:
0    36
1    13
Name: count, dtype: int64

Same fixed folds used for all K values.

FOLD 1
K=50  | Mean|r|=0.5839 | Cutoff=0.5391 | Acc=0.9000 | F1=0.9033 | Runtime=0.0024s
K=100 | Mean|r|=0.5534 | Cutoff=0.5108 | Acc=0.9000 | F1=0.9033 | Runtime=0.0036s
K=200 | Mean|r|=0.5171 | Cutoff=0.4557 | Acc=0.9000 | F1=0.8933 | Runtime=0.0030s
K=500 | Mean|r|=0.4504 | Cutoff=0.3656 | Acc=0.9000 | F1=0.9033 | Runtime=0.0049s

FOLD 2
K=50  | Mean|r|=0.5921 | Cutoff=0.5598 | Acc=1.0000 | F1=1.0000 | Runtime=0.0033s
K=100 | Mean|r|=0.5643 | Cutoff=0.5192 | Acc=1.0000 | F1=1.0000 | Runtime=0.0074s
K=200 | Mean|r|=0.5275 | Cutoff=0.4687 | Acc=1.0000 | F1=1.0000 | Runtime=0.0206s
K=500 | Mean|r|=0.4578 | Cutoff=0.3753 | Acc=1.0000 | F1=1.0000 | Runtime=0.0192s

FOLD 3
K=50

,Top_K,Mean_Pearson,Mean_Pearson_SD,Median_Pearson,Correlation_Cutoff,Correlation_Cutoff_SD,Accuracy_Mean,Accuracy_SD,F1_Mean,F1_SD,Jaccard_Stability,Runtime_Mean_sec,Runtime_SD_sec
0,50,0.6286,0.0387,0.6203,0.5950,0.0438,0.9178,0.0462,0.9175,0.0480,0.4742,0.0034,0.0022
1,100,0.6018,0.0409,0.5941,0.5593,0.0422,0.9178,0.0845,0.9171,0.0831,0.5028,0.0060,0.0016
2,200,0.5679,0.0433,0.5591,0.5126,0.0477,0.9600,0.0548,0.9593,0.0558,0.5598,0.0168,0.0171
3,500,0.5054,0.0480,0.4931,0.4267,0.0522,0.9000,0.1225,0.8997,0.1190,0.6056,0.0082,0.0062




PUBLICATION-READY TABLE


,Top-K Genes,Mean |r|,Correlation Cutoff,Accuracy (mean ± SD),F1-score (mean ± SD),Jaccard Stability,Runtime (s)
0,50,0.629,0.595,0.918 ± 0.046,0.917 ± 0.048,0.474,0.0034
1,100,0.602,0.559,0.918 ± 0.084,0.917 ± 0.083,0.503,0.0060
2,200,0.568,0.513,0.960 ± 0.055,0.959 ± 0.056,0.560,0.0168
3,500,0.505,0.427,0.900 ± 0.122,0.900 ± 0.119,0.606,0.0082



Saved successfully:
DLBCL_Sensitivity_Fold_Level.csv
DLBCL_Sensitivity_Detailed.csv
DLBCL_Sensitivity_Publication.csv

LATEX:
\begin{tabular}{rrrllrr}
\toprule
Top-K Genes & Mean |r| & Correlation Cutoff & Accuracy (mean ± SD) & F1-score (mean ± SD) & Jaccard Stability & Runtime (s) \\
\midrule
50 & 0.629000 & 0.595000 & 0.918 ± 0.046 & 0.917 ± 0.048 & 0.474000 & 0.003400 \\
100 & 0.602000 & 0.559000 & 0.918 ± 0.084 & 0.917 ± 0.083 & 0.503000 & 0.006000 \\
200 & 0.568000 & 0.513000 & 0.960 ± 0.055 & 0.959 ± 0.056 & 0.560000 & 0.016800 \\
500 & 0.505000 & 0.427000 & 0.900 ± 0.122 & 0.900 ± 0.119 & 0.606000 & 0.008200 \\
\bottomrule
\end{tabular}

